In [ ]:
# ---------------------------------------------------------
# SETTING UP FOLDERS
# ---------------------------------------------------------

from pathlib import Path

# We define a "Base" directory where all our project data will live.
# Use the current working directory so the notebooks work regardless of drive letter.
# If the notebooks are being run from a folder named "notebooks", use its parent as the project base.
_CWD = Path.cwd()
BASE = _CWD.parent if _CWD.name.lower() == "notebooks" else _CWD # Base is the folder you are currently in

# Now we create specific sub-folders inside our BASE folder.
DIR_RAW = BASE / "data_raw"   # Where we might save raw HTML files
DIR_PDFS = BASE / "pdfs"      # Where downloaded PDFs will go
DIR_OUT = BASE / "outputs"    # Where final processed data goes
DIR_LOG = BASE / "logs"       # Where we save error logs to track issues

# This loop goes through our list of folders.
for d in [DIR_RAW, DIR_PDFS, DIR_OUT, DIR_LOG]:
    # mkdir stands for "make directory". 
    # parents=True means "create any missing parent folders".
    # exist_ok=True means "don't crash if the folder already exists".
    d.mkdir(parents=True, exist_ok=True)

BASE

WindowsPath('D:/MYSQL-PYTHON-DATA')

In [ ]:
import os, re, json, time, random, hashlib, html as html_lib
from datetime import datetime, date
from decimal import Decimal, InvalidOperation
from typing import Optional


# dotenv lets us load hidden passwords from a secret file named ".env"
from dotenv import load_dotenv
import pymysql 

# This actually loads the secret variables from the .env file in our BASE folder.
load_dotenv(BASE / ".env")

True

In [ ]:
# ---------------------------------------------------------
# UTILITY FUNCTIONS (Helper tools)
# ---------------------------------------------------------

# This is useful for checking if we've already downloaded a file before.
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_text(s):
    # Text needs to be encoded into bytes before we can hash it.
    return hashlib.sha256(s.encode("utf-8", errors="ignore")).hexdigest()

# This safely combines a base website URL with a specific page link.
# e.g., urljoin("http://website.com", "/page1") -> "http://website.com/page1"
def urljoin(base, href):
    from urllib.parse import urljoin as _uj
    return _uj(base, href)

# Web scrapers use this to pause for a random amount of seconds (2 to 6).
# This makes the bot act more like a "human" so the website doesn't block it.
def sleep_human(min_s=2.0, max_s=6.0):
    time.sleep(random.uniform(min_s, max_s))

# This cleans up messy text scraped from the internet.
def norm_text(s):
    if not s:
        return ""
    s = html_lib.unescape(s)          # Fixes weird HTML characters
    s = s.replace("\xa0", " ")        # Replaces weird non-breaking spaces with normal spaces
    s = re.sub(r"\s+", " ", s).strip() # Squashes multiple spaces into just one
    return s

# This specifically cleans up names (like Company names).
def norm_name(s):
    s = norm_text(s).lower()          # Clean it and make it all lowercase
    s = s.replace("„", '"').replace("“", '"').replace("”", '"') # Standardize quotes
    s = re.sub(r"[’'`]", "'", s)      # Standardize apostrophes
    # Remove any characters that aren't letters, numbers, or basic punctuation.
    # (Notice it includes Lithuanian letters like ąčęėįšųūž!)
    s = re.sub(r"[^0-9a-zA-Ząčęėįšųūž\"'()\-.,/& ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip() # Squash extra spaces again
    return s

# Turns a string like "2023-10-31" into a real Python date object.
def parse_iso_date(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d").date()
    except Exception:
        return None # If it fails (bad formatting), just return None instead of crashing

# Turns European currency strings (like "1 000,50") into a Python Decimal number.
def parse_decimal_eur(s):
    s = s.strip().replace(" ", "").replace("\u202f", "") # Remove spaces
    s = s.replace(",", ".")                              # Change comma to a decimal point
    try:
        return Decimal(s)
    except InvalidOperation:
        return None

In [ ]:
# ---------------------------------------------------------
# DATABASE FUNCTIONS
# ---------------------------------------------------------

def db_connect():
    # We pull the credentials from the .env file using os.getenv
    return pymysql.connect(
        host=os.getenv("MYSQL_HOST"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        user=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        database=os.getenv("MYSQL_DB"),
        autocommit=False  # We want to manually commit changes after each transaction
    )

def db_exec(cur, sql: str, params: tuple = ()):
    # This calls the cursor.execute() function 
    cur.execute(sql, params)

def upsert_raw_page(cur, url, status, html_bytes, extracted: Optional[dict] = None):

    h = sha256_bytes(html_bytes)
    html_text = html_bytes.decode("utf-8", errors="ignore")
    extracted_json = json.dumps(extracted or {}, ensure_ascii=False)
    
    db_exec(cur, """
        INSERT INTO raw_page (url, sha256, http_status, html, extracted)
        VALUES (%s, %s, %s, %s, CAST(%s AS JSON))
        ON DUPLICATE KEY UPDATE
          sha256=VALUES(sha256),
          http_status=VALUES(http_status),
          html=VALUES(html),
          extracted=VALUES(extracted),
          fetched_at=CURRENT_TIMESTAMP
    """, (url, h, status, html_text, extracted_json))

def upsert_raw_case_row(cur, row):
    payload_json = json.dumps(row, ensure_ascii=False, default=str)
    
    # ensure_ascii=False: Keeps non-English characters (like ą, č, ę) readable 
    # instead of converting them to Unicode escape sequences (e.g., \u0105).
 
    # default=str: Prevents crashes by converting non-JSON-serializable objects 
    # (like datetime) into strings instead of throwing a TypeError.

    # We pass a long tuple of data to match the %s placeholders
    db_exec(cur, """
        INSERT INTO raw_case_row
          (year, case_type, source_list_url, list_page_updated_at, company_name_raw, subject_raw,
            pdf_url, pdf_url_hash, link_text, case_no_guess, decision_date_guess, payload)
        VALUES
          (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,CAST(%s AS JSON))
        ON DUPLICATE KEY UPDATE
          company_name_raw=VALUES(company_name_raw),
          subject_raw=VALUES(subject_raw),
          link_text=VALUES(link_text),
          case_no_guess=VALUES(case_no_guess),
          decision_date_guess=VALUES(decision_date_guess),
          list_page_updated_at=VALUES(list_page_updated_at),
          payload=VALUES(payload),
          scraped_at=CURRENT_TIMESTAMP
    """, (
        row["year"], row["case_type"], row["source_list_url"], row.get("list_page_updated_at"),
        row["company_name_raw"], row.get("subject_raw"),
        row["pdf_url"], row["pdf_url_hash"],
        row.get("link_text"),
        row.get("case_no_guess"), row.get("decision_date_guess"),
        payload_json
    ))
    
    # In pymysql, cur.fetchone() returns a tuple (e.g., (5,)). result[0] gets the ID.
    db_exec(cur, "SELECT row_id FROM raw_case_row WHERE pdf_url_hash=%s", (row["pdf_url_hash"],))
    result = cur.fetchone()
    return int(result[0]) if result else 0

def upsert_raw_pdf(cur, row_id, pdf_url, pdf_url_hash, sha, local_path, http_status):
    db_exec(cur, """
        INSERT INTO raw_pdf (row_id, pdf_url, pdf_url_hash, sha256, local_path, http_status)
        VALUES (%s,%s,%s,%s,%s,%s)
        ON DUPLICATE KEY UPDATE
          row_id=COALESCE(VALUES(row_id), row_id),
          sha256=VALUES(sha256),
          local_path=VALUES(local_path),
          http_status=VALUES(http_status),
          downloaded_at=CURRENT_TIMESTAMP
    """, (row_id, pdf_url, pdf_url_hash, sha, local_path, http_status))

#for checking if the columns actually exist in the db
def get_table_columns(cur, table_name):
    """Returns the current column names for a table as a lowercase set."""
    db_exec(cur, f"SHOW COLUMNS FROM `{table_name}`")
    return {str(row[0]).lower() for row in cur.fetchall()}

def upsert_raw_pdf_text(cur, sha, page_count, text):
    """
    Writes extracted PDF text into raw_pdf_text using only the columns that
    exist in the current schema.
    """
    cols = get_table_columns(cur, "raw_pdf_text")

    text_col = "text" if "text" in cols else ("pdf_text" if "pdf_text" in cols else None)
    if text_col is None:
        raise RuntimeError("raw_pdf_text table is missing both 'text' and 'pdf_text' columns")

    insert_columns = ["sha256"]
    insert_values = [sha]
    update_parts = []

    if "page_count" in cols:
        insert_columns.append("page_count")
        insert_values.append(page_count)
        update_parts.append("page_count=VALUES(page_count)")

    insert_columns.append(text_col)
    insert_values.append(text)
    update_parts.append(f"{text_col}=VALUES({text_col})")

    if "extracted_at" in cols:
        update_parts.append("extracted_at=CURRENT_TIMESTAMP")

    sql = f"""
        INSERT INTO raw_pdf_text ({", ".join(insert_columns)})
        VALUES ({", ".join(["%s"] * len(insert_columns))})
        ON DUPLICATE KEY UPDATE
          {", ".join(update_parts)}
    """
    db_exec(cur, sql, tuple(insert_values))
